### Creating an interactive dashboard that visualizes and compares the performance of machine learning models using key evaluation metrics such as Accuracy, Precision, Recall, and F1-score.This task focuses on model evaluation, data visualization, and user interaction, which are essential for real-world AI projects.

Step 1: import all the required libraries

In [16]:
# importing necessary libraries for data manipulation, model training, and evaluation

#core libraries
import pandas as pd
import numpy as np
import nltk   
import re

# File handling and model saving
import pickle
import os

# Machine learning tools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

# Evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Import NLP tools
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# download necessary NLTK resources
# These downloads are required for tokenization, stopwords, POS tagging, etc.
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

# Preprocessing and warnings
from sklearn.preprocessing import LabelEncoder
from utils import preprocess_text


# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


[nltk_data] Downloading package punkt to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Saif Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Step 2: Loading the dataset.

In [17]:
# laod dataset function to read the CSV file and return a DataFrame

def load_dataset(file_path):
    """
    Load dataset from CSV file
    
    Parameters:
        file_path (str): Path to the CSV file
    
    Returns:
        DataFrame: Loaded dataset
    """
    print("=" * 50)
    print("LOADING DATASET")
    print("=" * 50)
    
    df = pd.read_csv(file_path)
    print(f"✅ Dataset loaded from: {file_path}")
    print(f"   Shape: {df.shape}")
    
    return df

# Execute the function
df = load_dataset('news_dataset.csv')

LOADING DATASET
✅ Dataset loaded from: news_dataset.csv
   Shape: (9900, 2)


Step 3: Perform some basic operations like removing duplictaes and missing values

In [18]:
# perform data cleaning before preprocessing and training the models
def clean_dataset(df):
    """
    Clean the dataset by handling missing values, duplicates, and filtering reviews
    """
    # checking for missing values
    print("Missing Values:")
    print(df.isnull().sum())
    
    # since we got some missing values in the Text column, we will drop those rows
    df.dropna(subset=['Text'], inplace=True)
    
    #again check for missing values
    print("\nMissing Values After Cleaning:")
    print(df.isnull().sum())
    
    #checking for duplicates
    print(f"\nDuplicate rows found: {df.duplicated().sum()}")
    
    #we also got some duplicate rows, we will drop those as well
    df.drop_duplicates(inplace=True)
    
    #again check for duplicates
    print(f"Duplicates after removal: {df.duplicated().sum()}")
    
    #checking the head of the dataset after preprocessing
    print("\nFirst 5 rows after cleaning:")
    print(df.head())
    

    #checking the shape of the dataset after cleaning
    print(f"\nShape after cleaning: {df.shape}")
    
    return df

# Clean dataset (before preprocessing)
df = clean_dataset(df)

Missing Values:
Text     0
label    0
dtype: int64

Missing Values After Cleaning:
Text     0
label    0
dtype: int64

Duplicate rows found: 35
Duplicates after removal: 0

First 5 rows after cleaning:
                                                Text label
0   Top Trump Surrogate BRUTALLY Stabs Him In The...  Fake
1  U.S. conservative leader optimistic of common ...  Real
2  Trump proposes U.S. tax overhaul, stirs concer...  Real
3   Court Forces Ohio To Allow Millions Of Illega...  Fake
4  Democrats say Trump agrees to work on immigrat...  Real

Shape after cleaning: (9865, 2)


Step 4: Applying preprocess text.

In [19]:
# preprocess the text data using advance text cleaning techniques to improve the performance of the models
# Load stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Advanced NLP preprocessing pipeline:
    - Lowercasing
    - Remove URLs and HTML tags
    - Normalize currency and numbers
    - Remove special characters
    - Tokenization (NLTK)
    - POS tagging
    - Lemmatization with POS
    - Stopword removal
    - Remove short tokens
    - Clean extra spaces
    """
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)
    
    # 3. Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    
    # 4. Normalize currency (e.g., $100.50 → money)
    text = re.sub(r'\$\s?\d+(\.\d+)?', ' money ', text)
    
    # 5. Normalize numbers (e.g., 123 → number)
    text = re.sub(r'\d+(\.\d+)?', ' number ', text)
    
    # 6. Remove special characters & punctuation
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 7. Tokenization using NLTK
    tokens = word_tokenize(text)
    
    # 8. POS tagging
    pos_tags = pos_tag(tokens)
    
    # 9. Helper function for POS conversion (inside function as requested)
    def get_wordnet_pos(tag):
        if tag.startswith('J'):
            return 'a'  # adjective
        elif tag.startswith('V'):
            return 'v'  # verb
        elif tag.startswith('N'):
            return 'n'  # noun
        elif tag.startswith('R'):
            return 'r'  # adverb
        else:
            return 'n'
    
    # 10. Lemmatization + stopword removal + short word filtering
    cleaned_words = []
    for word, tag in pos_tags:
        if word not in stop_words and len(word) > 2:
            pos = get_wordnet_pos(tag)
            lemma = lemmatizer.lemmatize(word, pos)
            cleaned_words.append(lemma)
    
    # 11. Join tokens
    text = ' '.join(cleaned_words)
    
    # 12. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# apply the preprocessing function to the reviews
print("Applying text preprocessing...")
df['Text'] = df['Text'].apply(preprocess_text)

# Remove any empty strings after preprocessing
df = df[df['Text'] != ""]

# check the length and sample of the dataset after preprocessing
print("Text preprocessing completed!")
print(f"Number of reviews after preprocessing: {len(df)}")
print("\nSample preprocessed text:")
print(df['Text'].iloc[0][:200] + "...")

Applying text preprocessing...
Text preprocessing completed!
Number of reviews after preprocessing: 9865

Sample preprocessed text:
top trump surrogate brutally stab back pathetic video look though republican presidential candidate donald trump lose support even within rank know thing get bad even top surrogate start turn exactly ...


Step 5: Prepare the data for training and testing.

In [20]:
# Prepare features using TF-IDF and labels using LabelEncoder
def prepare_features(df):
    """
    Prepare features using TF-IDF and labels using LabelEncoder
    """
    # Split the dataset into features and target variable
    X = df['Text']  # Features (Text)
    y = df['label']  # Target variable (label)
    
    # Convert the target variable into numerical format using LabelEncoder
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    # See the mapping of the labels
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    print("Label Mapping:", label_mapping)
    
    # Check the shape of the features and target variable
    print("Features shape:", X.shape)
    print("Target variable shape:", y_encoded.shape)

    # First split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, 
        test_size=0.2, 
        random_state=42, 
        stratify=y_encoded
    )
    
    print(f"\nTraining set size: {X_train.shape[0]} samples")
    print(f"Test set size: {X_test.shape[0]} samples")
    
    # Apply TF-IDF vectorization AFTER split (fit on training data only)
    tfidf_vectorizer = TfidfVectorizer(max_features=5000)  # Limit to top 5000 features
    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)  # Fit and transform on training data
    X_test_tfidf = tfidf_vectorizer.transform(X_test)        # Only transform on test data
    
    print(f"\nTraining TF-IDF shape: {X_train_tfidf.shape}")
    print(f"Test TF-IDF shape: {X_test_tfidf.shape}")
    
    return X_train_tfidf, X_test_tfidf, y_train, y_test, label_encoder, tfidf_vectorizer

# Prepare features
X_train_tfidf, X_test_tfidf, y_train, y_test, label_encoder, tfidf_vectorizer = prepare_features(df)

Label Mapping: {'Fake': 0, 'Real': 1}
Features shape: (9865,)
Target variable shape: (9865,)

Training set size: 7892 samples
Test set size: 1973 samples

Training TF-IDF shape: (7892, 5000)
Test TF-IDF shape: (1973, 5000)


Step 6: Train and Evaluate the models.

In [21]:
# Function to train, evaluate models and create results
def train_and_evaluate_models(X_train_tfidf, X_test_tfidf, y_train, y_test):
    """
    Train multiple models, evaluate them, and return results
    
    Parameters:
        X_train_tfidf (array): Training features
        X_test_tfidf (array): Test features
        y_train (array): Training labels
        y_test (array): Test labels
    
    Returns:
        tuple: (models_dict, results_dataframe, predictions_dict)
    """
    # Initialize models
    print("=" * 50)
    print("TRAINING AND EVALUATING MODELS")
    print("=" * 50)
    
    lr = LogisticRegression(max_iter=1000)   # Logistic Regression
    svm = SVC(probability=True)              # SVM (probability needed for ROC)
    nb = MultinomialNB()                     # Naive Bayes
    
    # Train models
    print("\nTraining models...")
    lr.fit(X_train_tfidf, y_train)
    print("✅ Logistic Regression trained")
    
    svm.fit(X_train_tfidf, y_train)
    print("✅ SVM trained")
    
    nb.fit(X_train_tfidf, y_train)
    print("✅ Naive Bayes trained")
    
    # Store models in dictionary
    models = {
        "Logistic Regression": lr,
        "SVM": svm,
        "Naive Bayes": nb
    }
    
    results = []        # Store evaluation results
    preds_dict = {}     # Store predictions for each model
    
    # Evaluate each model
    print("\nEvaluating models...")
    for name, model in models.items():
        
        # Predict on test data
        preds = model.predict(X_test_tfidf)
        
        # Save predictions
        preds_dict[name] = preds
        
        # Calculate evaluation metrics
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, average='weighted')
        rec = recall_score(y_test, preds, average='weighted')
        f1 = f1_score(y_test, preds, average='weighted')
        
        # Store results
        results.append([name, acc, prec, rec, f1])
        
        # Print metrics
        print(f"\n{name}:")
        print(f"   Accuracy:  {acc:.4f}")
        print(f"   Precision: {prec:.4f}")
        print(f"   Recall:    {rec:.4f}")
        print(f"   F1-score:  {f1:.4f}")
    
    # Create results table
    results_df = pd.DataFrame(results, columns=[
        "Model", "Accuracy", "Precision", "Recall", "F1-score"
    ])
    
    return models, results_df, preds_dict

# Call the function
models, results_df, preds_dict = train_and_evaluate_models(X_train_tfidf, X_test_tfidf, y_train, y_test)

# Display results
print("\n" + "=" * 50)
print("FINAL RESULTS")
print("=" * 50)
results_df

TRAINING AND EVALUATING MODELS

Training models...
✅ Logistic Regression trained
✅ SVM trained
✅ Naive Bayes trained

Evaluating models...

Logistic Regression:
   Accuracy:  0.9934
   Precision: 0.9934
   Recall:    0.9934
   F1-score:  0.9934

SVM:
   Accuracy:  0.9954
   Precision: 0.9954
   Recall:    0.9954
   F1-score:  0.9954

Naive Bayes:
   Accuracy:  0.9640
   Precision: 0.9641
   Recall:    0.9640
   F1-score:  0.9640

FINAL RESULTS


,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.993411,0.993416,0.993411,0.993411
1,SVM,0.995438,0.995439,0.995438,0.995438
2,Naive Bayes,0.964014,0.964077,0.964014,0.964016


In [22]:
# Function to save all models and artifacts
def save_all_models(results_df, label_encoder, models, tfidf_vectorizer, y_test, preds_dict, X_test_tfidf, models_dir='models'):
    """
    Save all models, vectorizer, encoder, and evaluation results
    
    Parameters:
        results_df (DataFrame): Results table
        label_encoder: LabelEncoder object
        models (dict): Dictionary of trained models
        tfidf_vectorizer: TF-IDF vectorizer
        y_test (array): Test labels
        preds_dict (dict): Predictions dictionary
        X_test_tfidf (array): Test features
        models_dir (str): Directory to save artifacts
    """
    # Create 'models' folder if it doesn't exist
    os.makedirs(models_dir, exist_ok=True)
    
    # Save results table
    results_df.to_csv(f"{models_dir}/results.csv", index=False)
    
    # Save models and required objects
    pickle.dump(label_encoder, open(f"{models_dir}/label_encoder.pkl", "wb"))
    pickle.dump(models, open(f"{models_dir}/models.pkl", "wb"))
    pickle.dump(tfidf_vectorizer, open(f"{models_dir}/tfidf_vectorizer.pkl", "wb"))
    pickle.dump(y_test, open(f"{models_dir}/y_test.pkl", "wb"))
    pickle.dump(preds_dict, open(f"{models_dir}/preds.pkl", "wb"))
    pickle.dump(X_test_tfidf, open(f"{models_dir}/X_test_tfidf.pkl", "wb"))
    
    print("All files saved successfully!")

# Call the function
save_all_models(results_df, label_encoder, models, tfidf_vectorizer, y_test, preds_dict, X_test_tfidf)

All files saved successfully!
